In [9]:
import torch
import torchvision
import numpy
import pathlib

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [11]:
class MapDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

In [12]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    torchvision.transforms.ToTensor(),
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
])

In [13]:
dateset_path = pathlib.Path("C:\\Users\\Emir\\Desktop\\Desktop\\TMP\\code\\AI\\garbage classifier\\dataset")

base_dataset = torchvision.datasets.ImageFolder(root=dateset_path)
dataset_train, dataset_test = torch.utils.data.random_split(dataset=base_dataset, lengths=[0.8,0.2]) 

dataset_train = MapDataset(dataset_train, transform=train_transform)
dataset_test = MapDataset(dataset_test, transform=test_transform)

dataloader_train = torch.utils.data.DataLoader(dataset=dataset_train, batch_size=128, shuffle=True, num_workers=0, pin_memory=True) 
dataloader_test = torch.utils.data.DataLoader(dataset=dataset_test, batch_size=128, shuffle=False, num_workers=0, pin_memory=True) 

In [14]:
class GarbageClassifierModel(torch.nn.Module): 
    def __init__(self) -> None:
        super().__init__()
        self.convolution_layer()
        self.flatten = torch.nn.Flatten()
        self.classification_layer()

    def convolution_layer(self): 
        self.conv_block1 = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv_block2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64), 
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv_block3 = torch.nn.Sequential(
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv_block4 = torch.nn.Sequential(
            torch.nn.Conv2d(128, 256, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )

        self.pool_final = torch.nn.AdaptiveAvgPool2d((7, 7))

    def classification_layer(self):
        self.classification = torch.nn.Sequential(
            torch.nn.Linear(2*6272, 512), 
            torch.nn.ReLU(), 
            torch.nn.Dropout(0.5), 

            torch.nn.Linear(512, 128), 
            torch.nn.ReLU(), 

            torch.nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.pool_final(x)
        x = self.flatten(x) 
        return self.classification(x)
        



In [15]:
model = GarbageClassifierModel().to(device)

loss = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [16]:
best_accuracy = 0.0
for epoch in range(50): 
    model.train()
    running_loss = 0.0

    for images, labels in dataloader_train: 
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        train_loss = loss(outputs, labels)
        train_loss.backward()
        optimizer.step()

        running_loss += train_loss.item()
    
    epoch_loss = running_loss / len(dataloader_train)
    print(f"Epoch {epoch+1} - Training Loss: {epoch_loss:.4f}")

    model.eval()
    with torch.no_grad(): 
        correct = 0
        total = 0
        for images, labels in dataloader_test: 
            total += len(labels)

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            test = loss(outputs, labels)

            output_labels = torch.argmax(outputs, dim=1)

            correct_guesses = output_labels == labels
            correct += torch.count_nonzero(correct_guesses).item()

        accuracy = correct/total * 100
        print(f"Accuracy = {accuracy}%")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), 'best_garbage_model.pth')
            print("--- Model Saved! ---")


Epoch 1 - Training Loss: 1.5488
Accuracy = 28.118811881188122%
--- Model Saved! ---
Epoch 2 - Training Loss: 1.2523
Accuracy = 37.62376237623762%
--- Model Saved! ---
Epoch 3 - Training Loss: 1.1066
Accuracy = 56.43564356435643%
--- Model Saved! ---
Epoch 4 - Training Loss: 1.0064
Accuracy = 60.396039603960396%
--- Model Saved! ---
Epoch 5 - Training Loss: 0.9600
Accuracy = 71.28712871287128%
--- Model Saved! ---
Epoch 6 - Training Loss: 0.8817
Accuracy = 68.11881188118812%
Epoch 7 - Training Loss: 0.8371
Accuracy = 69.9009900990099%
Epoch 8 - Training Loss: 0.7883
Accuracy = 73.46534653465346%
--- Model Saved! ---
Epoch 9 - Training Loss: 0.7668
Accuracy = 70.49504950495049%
Epoch 10 - Training Loss: 0.7192
Accuracy = 71.88118811881188%
Epoch 11 - Training Loss: 0.6612
Accuracy = 75.84158415841584%
--- Model Saved! ---
Epoch 12 - Training Loss: 0.6431
Accuracy = 76.23762376237624%
--- Model Saved! ---
Epoch 13 - Training Loss: 0.6190
Accuracy = 78.61386138613862%
--- Model Saved! ---


In [19]:
saved_model = GarbageClassifierModel().to(device)
state_dict = torch.load('best_garbage_model.pth')
saved_model.load_state_dict(state_dict)

saved_model.eval()
with torch.no_grad(): 
    correct = 0
    total = 0
    for images, labels in dataloader_test: 
        total += len(labels)

        images = images.to(device)
        labels = labels.to(device)

        outputs = saved_model(images)
        test = loss(outputs, labels)

        output_labels = torch.argmax(outputs, dim=1)

        correct_guesses = output_labels == labels
        correct += torch.count_nonzero(correct_guesses).item()

    accuracy = correct/total * 100
    print(f"Accuracy = {accuracy}%")



Accuracy = 84.55445544554455%
